In [ ]:
import torch,transformers
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
print('GPU:',torch.cuda.get_device_name(0) if cap else 'none','sm',cap,'| transformers',transformers.__version__)
print('GPU_OK' if (cap and cap[0]>=7) else 'GPU needs T4 (set in Notebook options)')

In [ ]:
!pip -q install -U --no-deps transformers
!pip -q install -U peft trl datasets
print('deps ready')

In [ ]:
import os,glob,json
p=None
for cand in ["/kaggle/input/enilo-rehan-v1/train_v1.jsonl"]+glob.glob("/kaggle/input/**/train_v1.jsonl",recursive=True):
    if os.path.exists(cand): p=cand; break
assert p, 'dataset not attached to kernel'
print('dataset at:',p)
from datasets import Dataset
rows=[json.loads(l) for l in open(p)]
def norm(r):
    out=[]
    for m in r['messages']:
        if m.get('content') is None and m.get('tool_calls'):
            tc=m['tool_calls'][0]['function']
            m={'role':'assistant','content':f'<tool_call name="{tc["name"]}">{tc["arguments"]}</tool_call>'}
        out.append(m)
    return out
ds=Dataset.from_list([{'messages':norm(r)} for r in rows])
print('lessons:',len(ds))

In [ ]:
import torch
from trl import SFTConfig,SFTTrainer
from peft import LoraConfig
B='prithivMLmods/Qwen3.5-4B-Opus-Distilled-Heretic-Thinking-Multistage-SFT-v1.0'
pc=LoraConfig(r=16,lora_alpha=32,target_modules='all-linear',task_type='CAUSAL_LM')
cfg=SFTConfig(output_dir='/kaggle/working/run',per_device_train_batch_size=1,
 gradient_accumulation_steps=8,learning_rate=1.5e-4,num_train_epochs=3,logging_steps=5,
 max_length=1536,eos_token='<|im_end|>',save_strategy='no',report_to=[])
trk=SFTTrainer(model=B,args=cfg,train_dataset=ds,peft_config=pc)
print('trainer ready — model auto-loads bf16 with qwen3_5 template')

In [ ]:
trk.train(); print('train done')

In [ ]:
m=trk.model.merge_and_unload() if hasattr(trk.model,'merge_and_unload') else trk.model
o='/kaggle/working/enilo-rehan-v1'
m.save_pretrained(o); trk.processing_class.save_pretrained(o) if getattr(trk,'processing_class',None) else None
import os
tot=0
for r,_,fs in os.walk(o):
    for f in fs:
        s=os.path.getsize(os.path.join(r,f)); tot+=s
        print(f'\t{s//1_000_000}MB\t{f}')
print(f'TOTAL {tot/1e9:.2f}GB')